In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

cifar_data = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

print("CIFAR-10 loaded:", len(cifar_data))
print("Image shape:", cifar_data[0][0].shape)


In [ ]:
class ColorizationDataset(Dataset):
    def __init__(self, cifar_dataset):
        self.ds = cifar_dataset

    def __len__(self):
        return len(self.ds)

    def rgb_to_grayscale(self, img):
        w = torch.tensor([0.299, 0.587, 0.114], device=img.device).view(3,1,1)
        gray = (img * w).sum(dim=0, keepdim=True)
        return gray

    def __getitem__(self, idx):
        color_img, _ = self.ds[idx]
        gray_img = self.rgb_to_grayscale(color_img)
        return gray_img, color_img


In [ ]:
colorization_dataset = ColorizationDataset(cifar_data)

gray_img, color_img = colorization_dataset[0]
print("Grayscale image shape:", gray_img.shape)
print("Color image shape:", color_img.shape)

fig, axes = plt.subplots(1,2, figsize=(6,3))
axes[0].imshow(gray_img.squeeze(), cmap="gray")
axes[0].set_title("Grayscale (Input)")
axes[0].axis("off")

axes[1].imshow(color_img.permute(1,2,0))
axes[1].set_title("Color (Target)")
axes[1].axis("off")
plt.show()

In [ ]:
loader = DataLoader(colorization_dataset, batch_size=8, shuffle=True)
gray_batch, color_batch = next(iter(loader))
print("Batch grayscale shape:", gray_batch.shape)
print("Batch color shape:", color_batch.shape)